# Train RNNs on all sessions of real neural data

In [1]:
from utilities.utils import *
from utilities.RNN import *
import numpy as np
import scipy
import random
import torch
import pickle

# Load and prepare data

In [4]:
## load raw data
data = scipy.io.loadmat('data/experimental/data_all.mat')

## parameters for data prep
bin_size = 0.050 # [s]
padding = 1 # [s]
min_n_neurons = 3
smooth_type = 'gaussian'
smooth_width = 11

# filter sessions
n_sessions_all = data['data_all'].shape[1]
included_sessions = [i for i in range(n_sessions_all) if len(data['data_all'][0, i][-1]) >= min_n_neurons]
n_sessions = len(included_sessions)

# time warping
try:
    with open('data/experimental/data_warped.pkl', 'rb') as file:
        all_data = pickle.load(file)
        file.close()
except:
    all_data = {}
    for session in included_sessions:
        print('Preparing session', session + 1, '...')
        all_data[session] = get_warped_data(data, session, bin_size, padding, mean_correct_only=False)
    print('Done.')

# smoothing
for session in included_sessions:
    X, y_conc, y_choice, y_outcome, t = all_data[session]
    X_smooth = X.copy()
    for i in range(X_smooth.shape[0]):
        for j in range(X_smooth.shape[1]):
            X_smooth[i, j, :] = smooth(X_smooth[i, j, :], smooth_type, smooth_width)
    all_data[session] = (X_smooth, y_conc, y_choice, y_outcome, t)

## calculate additional variables
unique_stims = np.unique(all_data[0][1]) # same stimuli for all sessions
n_stims = len(unique_stims)
t = all_data[0][4] # same time for all sessions
n_bins = len(t)

## save model inputs
with open('data/model/experimental_input.pkl', 'wb') as file:
    pickle.dump({
        'parameters': {
            'bin_size': bin_size,
            'padding': padding,
            'smooth_type': smooth_type,
            'smooth_width': smooth_width
        },
        'all_data': all_data
    }, file)
    file.close()

# Model

In [ ]:
## hyperparameters

# architecture
input_size = 2 # stimulus will look like x = [%sucrose / 100, %NaCl / 100], so [0, 1] = NaCl and [1, 0] = sucrose
network_size_scale = 5.88 # ratio of total network size to number of constrained units
expansion_size_scale = 1 # ratio of 'input layer' size to network size
max_fr = 80 # hard cap on ReLU activation function that returns 'firing rates'
stim_start = 0
stim_end = 1.2
decision_pre_time = 0.1 # pre-decision time point window

# training
n_epochs = 2000
lr = 1e-2
early_stop = 1
normalize_loss = False
train_wi = False
train_si = True
train_h0 = True
train_b = True
train_wrec = True
train_wz = True
lambda_neural = 1
lambda_behavioral = 150
clip_gradient = 1.0

In [ ]:
## fit one model per session 

all_models = {}

for session in all_data:
    
    ## grab data
    X, y_conc, y_choice, y_outcome, t = all_data[session]

    ## session-specific variables
    n_trials, n_neurons, n_bins = X.shape
    t_stim_window = (t >= stim_start) & (t <= stim_end)
    t_D = t[-1] + bin_size / 2 - 1.0 # decision time point
    t_decision_window = (t > t_D - decision_pre_time) & (t <= t_D)
    t_target_window = (t < 0) | t_decision_window # window during which we care about behavioral output

    ## set up model inputs and outputs
    inputs = np.zeros((n_stims, n_bins, input_size))
    outputs = np.zeros((n_stims, n_bins, n_neurons + 1))
    for i_stim, stim in enumerate(unique_stims):
        inputs[i_stim, :, :] = np.tile( np.array([stim / 100, 1 - stim / 100]), (n_bins, 1) )
        trial_mask = (y_conc == stim) & (y_outcome == 1)
        for i_neuron in range(n_neurons):
            psth = X[trial_mask, i_neuron, :].mean(axis=0)
            outputs[i_stim, :, i_neuron] = psth
        choice = (1.0 if (stim > 50) else -1.0)
        outputs[i_stim, t_decision_window, -1] = choice

    # mixture stimulus is only present within [stim_start, stim_end]
    inputs[:, ~t_stim_window, :] = 0

    # convert to tensors
    inputs = torch.from_numpy(inputs).to(dtype=torch.float32)
    outputs = torch.from_numpy(outputs).to(dtype=torch.float32)

    # we only care about the decision variable before the stimulus (should be 0) and during decision window
    mask_train = torch.ones(outputs.shape).to(dtype=torch.float32)
    mask_train[:, ~t_target_window, -1] = 0
    
    ## other model parameters
    network_size = int(round(network_size_scale * n_neurons))
    input_expansion_size = int(round(expansion_size_scale * network_size))

    ## initialize model
    np.random.seed(0) # for repeatability
    torch.manual_seed(0)
    random.seed(0)

    net = MyRNN(
        input_size=input_size, 
        observed_size=n_neurons,
        hidden_size=(network_size - n_neurons),
        input_expansion_size=input_expansion_size,  
        train_wi=train_wi,
        train_si=train_si,
        train_h0=train_h0,
        train_b=train_b,
        train_wrec=train_wrec,
        train_wz=train_wz,
        non_linearity=ReLUX(max_fr)
    )
    net_init = net.clone()

    ## train model
    train_no_accuracy(
        net, 
        inputs, 
        outputs, 
        mask_train, 
        n_epochs, 
        t_target_window,
        lambda_neural=lambda_neural,
        lambda_behavioral=lambda_behavioral,
        lambda_bias_correction=0,
        lr=lr, 
        batch_size=inputs.shape[0], 
        clip_gradient=clip_gradient, 
        early_stop=early_stop, 
        keep_best=True,  
        normalize_loss=normalize_loss,
        verbose=False
    )

    net.cpu()
    
    all_models[session] = net.clone()

In [ ]:
## save the data

# with open('data/model/all_models.pkl', 'wb') as file:
#     pickle.dump(all_models, file)
#     file.close()

# with open('data/model/hyperparameters.pkl', 'wb') as file:
#     hyperparameters = {
#         'input_size': input_size, 
#         'network_size_scale': network_size_scale,
#         'expansion_size_scale': expansion_size_scale,
#         'max_fr': max_fr,
#         'stim_start': stim_start,
#         'stim_end': stim_end,
#         'decision_pre_time': decision_pre_time,
#         'n_epochs': n_epochs,
#         'lr': lr,
#         'early_stop': early_stop,
#         'normalize_loss': normalize_loss,
#         'train_wi': train_wi,
#         'train_si': train_si,
#         'train_h0': train_h0,
#         'train_b': train_b,
#         'train_wrec': train_wrec,
#         'train_wz': train_wz,
#         'lambda_neural': lambda_neural,
#         'lambda_behavioral': lambda_behavioral,
#         'clip_gradient': clip_gradient
#     }
#     pickle.dump(hyperparameters, file)
#     file.close()